# Flight Delay Model — Full Training on Google Colab

### Before running:
1. Upload `data_new.zip` to your Google Drive (root or a folder)
2. Update `ZIP_PATH` below to match where you put it
3. Run all cells top to bottom
4. The last cell downloads the trained model files — save them to your `models/` folder

In [ ]:
# Cell 0 — Install dependencies
!pip install -q lightgbm meteostat psutil

In [ ]:
# Cell 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ── UPDATE THIS PATH to where you uploaded data_new.zip ──
ZIP_PATH = '/content/drive/MyDrive/data_new.zip'

import os
assert os.path.exists(ZIP_PATH), f'ZIP not found at {ZIP_PATH} — check the path above'
print('ZIP found:', ZIP_PATH)

In [ ]:
# Cell 2 — Extract zip
import zipfile, os

EXTRACT_DIR = '/content/data_new'
os.makedirs(EXTRACT_DIR, exist_ok=True)

print('Extracting... (this takes ~2 minutes)')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_DIR)

csv_files = []
for root, dirs, files in os.walk(EXTRACT_DIR):
    for f in files:
        if f.endswith('.csv'):
            csv_files.append(os.path.join(root, f))
csv_files = sorted(csv_files)
print(f'Found {len(csv_files)} CSV files')

In [ ]:
# Cell 3 — Imports & config
import gc, glob, warnings
import pandas as pd
import numpy as np
import joblib
import lightgbm as lgb
import psutil
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import classification_report, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

MODELS_DIR    = '/content/models'
WEATHER_CACHE = '/content/weather_cache'
os.makedirs(MODELS_DIR,    exist_ok=True)
os.makedirs(WEATHER_CACHE, exist_ok=True)

ROWS_PER_FILE = 21000
TOP_K         = 50
RANDOM_STATE  = 42

print('LightGBM:', lgb.__version__)
print(f'RAM available: {psutil.virtual_memory().available/1e9:.1f} GB')

In [ ]:
# Cell 4 — Load all BTS CSV files (~987K rows total)
USE_COLS = ['Year','Month','DayofMonth','DayOfWeek','FlightDate',
            'Reporting_Airline','Origin','Dest',
            'CRSDepTime','CRSArrTime','ArrDelay',
            'Distance','Cancelled','Diverted']

chunks = []
for f in csv_files:
    try:
        tmp = pd.read_csv(f, low_memory=False, usecols=USE_COLS, nrows=ROWS_PER_FILE)
        chunks.append(tmp)
    except Exception as e:
        print(f'  Skipping {f}: {e}')

df = pd.concat(chunks, ignore_index=True)
del chunks; gc.collect()

print(f'Loaded: {len(df):,} rows x {df.shape[1]} cols')
print(f'Years: {sorted(df["Year"].unique())}')
print(f'RAM used: {psutil.Process(os.getpid()).memory_info().rss/1e9:.2f} GB')

In [ ]:
# Cell 5 — Clean
df = df[(df['Cancelled'] != 1) & (df['Diverted'] != 1)].copy()
df['ArrDelay'] = pd.to_numeric(df['ArrDelay'], errors='coerce')
df = df[df['ArrDelay'].notna()].copy()
df['Origin']     = df['Origin'].astype(str).str.strip().str.upper()
df['Dest']       = df['Dest'].astype(str).str.strip().str.upper()
df['carrier']    = df['Reporting_Airline'].astype(str).str.strip().str.upper()
df['FlightDate'] = pd.to_datetime(df['FlightDate'], errors='coerce')

print(f'After cleaning: {len(df):,} rows')
print('Carriers:', sorted(df['carrier'].unique()))

In [ ]:
# Cell 6 — Feature engineering
def hhmm_to_hour(x):
    try: return int(float(x)) // 100
    except: return np.nan

df['dep_hour']  = df['CRSDepTime'].apply(hhmm_to_hour)
df['arr_hour']  = df['CRSArrTime'].apply(hhmm_to_hour)
df['month']     = df['Month'].astype(float)
df['dayofweek'] = df['DayOfWeek'].astype(float)
df['distance']  = pd.to_numeric(df['Distance'], errors='coerce')

top_orig_list = df['Origin'].value_counts().nlargest(TOP_K).index.tolist()
top_dest_list = df['Dest'].value_counts().nlargest(TOP_K).index.tolist()
df['origin_topK'] = df['Origin'].where(df['Origin'].isin(top_orig_list), other='OTHER')
df['dest_topK']   = df['Dest'].where(df['Dest'].isin(top_dest_list),   other='OTHER')

df['DELAY_MINUTES'] = df['ArrDelay']
df['DELAY_CLASS']   = df['DELAY_MINUTES'].apply(
    lambda x: 0 if x <= 15 else (1 if x <= 59 else 2)
).astype(int)

print('Delay class distribution:')
print(df['DELAY_CLASS'].value_counts().sort_index())

In [ ]:
# Cell 7 — Fetch weather (cached per airport)
# This cell uploads airports.csv from Drive — update the path if needed
from meteostat import Stations, Hourly

# Upload airports.csv to Drive and update path, OR let it download below
AIRPORTS_DRIVE = '/content/drive/MyDrive/airports.csv'
if os.path.exists(AIRPORTS_DRIVE):
    airports_df = pd.read_csv(AIRPORTS_DRIVE)
else:
    # Fallback: minimal coords for top 50 US airports embedded
    print('airports.csv not found on Drive — please upload it alongside data_new.zip')
    raise FileNotFoundError('Upload Data/airports.csv to Google Drive')

airports_df.columns = [c.lower() for c in airports_df.columns]
airports_df['iata'] = airports_df['iata'].astype(str).str.strip().str.upper()

all_airports   = list(set(top_orig_list + top_dest_list) - {'OTHER'})
airport_coords = airports_df[airports_df['iata'].isin(all_airports)].set_index('iata')[['lat','lon']]

START = datetime(2022, 1, 1)
END   = datetime(2025, 11, 30)

weather_frames = []
print(f'Fetching weather for {len(airport_coords)} airports...')
for iata, row in airport_coords.iterrows():
    cache_file = f'{WEATHER_CACHE}/{iata}.parquet'
    if os.path.exists(cache_file):
        weather_frames.append(pd.read_parquet(cache_file))
        continue
    try:
        stations = Stations().nearby(row['lat'], row['lon']).fetch(1)
        if stations.empty: continue
        wx = Hourly(stations.index[0], START, END).fetch()
        if wx.empty: continue
        wx = wx[['temp','wspd','prcp','snow','coco']].reset_index()
        wx.columns = ['time','temp','wspd','prcp','snow','coco']
        wx['iata'] = iata
        wx['date'] = wx['time'].dt.date.astype(str)
        wx['hour'] = wx['time'].dt.hour
        wx = wx.drop(columns=['time'])
        wx.to_parquet(cache_file, index=False)
        weather_frames.append(wx)
    except Exception as e:
        print(f'  {iata}: {e}')

weather_df = pd.concat(weather_frames, ignore_index=True)
print(f'Weather loaded: {len(weather_df):,} hourly records')

In [ ]:
# Cell 8 — Merge weather onto flights
df['date'] = df['FlightDate'].dt.date.astype(str)
df = df.merge(
    weather_df.rename(columns={'iata': 'Origin'}),
    left_on=['Origin','date','dep_hour'],
    right_on=['Origin','date','hour'],
    how='left'
).drop(columns=['hour'], errors='ignore')

for col in ['temp','wspd','prcp','snow','coco']:
    df[col] = df[col].fillna(df[col].median())

print(f'Weather match rate: {df["wspd"].notna().mean():.1%}')
print(f'Shape after merge: {df.shape}')

In [ ]:
# Cell 9 — Train/test split
FEATURE_COLS = ['dep_hour','arr_hour','month','dayofweek','distance',
                'temp','wspd','prcp','snow','coco',
                'origin_topK','dest_topK','carrier']

X     = df[FEATURE_COLS].copy()
y_cls = df['DELAY_CLASS']
y_reg = df['DELAY_MINUTES']

X_train, X_test, y_train_cls, y_test_cls = train_test_split(
    X, y_cls, test_size=0.2, stratify=y_cls, random_state=RANDOM_STATE
)
y_train_reg = y_reg.loc[y_train_cls.index]
y_test_reg  = y_reg.loc[y_test_cls.index]

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

In [ ]:
# Cell 10 — Route average delay (computed on train only)
train_tmp = X_train[['origin_topK','dest_topK']].copy()
train_tmp['DELAY_MINUTES'] = y_train_reg.values

route_avg = (
    train_tmp.groupby(['origin_topK','dest_topK'])['DELAY_MINUTES']
    .mean().reset_index()
    .rename(columns={'DELAY_MINUTES': 'route_avg_delay'})
)
global_avg = float(train_tmp['DELAY_MINUTES'].mean())

def add_route_avg(X_df):
    out = X_df.merge(route_avg, on=['origin_topK','dest_topK'], how='left')
    out['route_avg_delay'] = out['route_avg_delay'].fillna(global_avg)
    return out

X_train = add_route_avg(X_train)
X_test  = add_route_avg(X_test)

FEATURE_COLS_FULL = FEATURE_COLS + ['route_avg_delay']
print(f'Global avg delay: {global_avg:.1f} min')

In [ ]:
# Cell 11 — Encode categoricals & fill numeric NaNs
CAT_COLS = ['origin_topK','dest_topK','carrier']
NUM_COLS = [c for c in FEATURE_COLS_FULL if c not in CAT_COLS]

num_medians = {}
for col in NUM_COLS:
    med = float(X_train[col].median())
    num_medians[col] = med
    X_train[col] = X_train[col].fillna(med)
    X_test[col]  = X_test[col].fillna(med)

enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[CAT_COLS] = enc.fit_transform(X_train[CAT_COLS]).astype(int)
X_test[CAT_COLS]  = enc.transform(X_test[CAT_COLS]).astype(int)

for col in CAT_COLS:
    X_train[col] = pd.Categorical(X_train[col])
    X_test[col]  = pd.Categorical(X_test[col])

X_train = X_train[FEATURE_COLS_FULL]
X_test  = X_test[FEATURE_COLS_FULL]

print(f'Final shape: {X_train.shape}')
print(f'RAM: {psutil.Process(os.getpid()).memory_info().rss/1e9:.2f} GB')

In [ ]:
# Cell 12 — Train LightGBM Classifier
clf = lgb.LGBMClassifier(
    n_estimators      = 500,
    num_leaves        = 63,
    learning_rate     = 0.05,
    class_weight      = 'balanced',
    min_child_samples = 20,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    random_state      = RANDOM_STATE,
    n_jobs            = -1,
    verbose           = -1
)
clf.fit(
    X_train, y_train_cls,
    eval_set=[(X_test, y_test_cls)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)

y_pred_cls = clf.predict(X_test)
print('\n=== Classification Report ===')
print(classification_report(y_test_cls, y_pred_cls,
      target_names=['On-time','Minor delay','Major delay']))

In [ ]:
# Cell 13 — Train LightGBM Regressor
reg = lgb.LGBMRegressor(
    n_estimators      = 500,
    num_leaves        = 63,
    learning_rate     = 0.05,
    min_child_samples = 20,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    random_state      = RANDOM_STATE,
    n_jobs            = -1,
    verbose           = -1
)
reg.fit(
    X_train, y_train_reg,
    eval_set=[(X_test, y_test_reg)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)

y_pred_reg = reg.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
r2   = r2_score(y_test_reg, y_pred_reg)
print(f'\nRMSE: {rmse:.2f} min  |  R²: {r2:.4f}')

In [ ]:
# Cell 14 — Save all model artifacts
FEATURE_COLS_BASE = ['dep_hour','arr_hour','month','dayofweek','distance',
                     'temp','wspd','prcp','snow','coco',
                     'origin_topK','dest_topK','carrier']

artifacts = {
    f'{MODELS_DIR}/rf_classifier_sample.joblib':  clf,
    f'{MODELS_DIR}/rf_regressor_sample.joblib':   reg,
    f'{MODELS_DIR}/ordinal_encoder.joblib':        enc,
    f'{MODELS_DIR}/top_orig.joblib':               top_orig_list,
    f'{MODELS_DIR}/top_dest.joblib':               top_dest_list,
    f'{MODELS_DIR}/route_avg_delay.joblib':        {'route_avg': route_avg, 'global_avg': global_avg},
    f'{MODELS_DIR}/preprocessor_sample.joblib':    {
        'feature_cols': FEATURE_COLS_FULL,
        'cat_cols':     CAT_COLS,
        'num_cols':     NUM_COLS,
        'num_medians':  num_medians,
    },
}

for path, obj in artifacts.items():
    joblib.dump(obj, path)
    print(f'  Saved {path}  ({os.path.getsize(path)/1e6:.1f} MB)')

# Save sample parquet for the app
sample_cols = FEATURE_COLS_BASE + ['DELAY_CLASS','DELAY_MINUTES']
df[sample_cols].head(10000).to_parquet(f'{MODELS_DIR}/processed_flights_sample.parquet', index=False)
print('\nAll artifacts saved!')

In [ ]:
# Cell 15 — Download all model files to your computer
# After downloading, replace the files in your local models/ folder
import shutil
from google.colab import files

# Zip all models into one file for easy download
shutil.make_archive('/content/models_full', 'zip', MODELS_DIR)
files.download('/content/models_full.zip')
print('Download started — unzip and replace your local models/ folder')